# Notebook to assess MCS statistics from MPAS aquaplanet model output.

### Main settings

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.ndimage import gaussian_filter1d
import seaborn as sns
import gc

In [ ]:
exp_names = ['CTL', 'CLIM_RAD_LW']
exp_tags = ['MPAS_CTL', 'MPAS_OffLW']
ntest = len(exp_names)

data_main = "./data/"
figdir = "./figures/"

pickle_path = "../../pickle_out/aquaplanet"

# P-class cluster settings
raw_ts_lat_bounds = (10, 20) # latitude range
# raw_ts_area_threshold_method = "percentile" # "fixed" or "percentile"
raw_ts_area_threshold_method = "fixed" # "fixed" or "percentile"
raw_ts_area_percentile_range = (95, np.inf) # lower/upper percentiles
raw_ts_min_area_km2 = 100 # fixed-mode minimum area threshold
raw_ts_min_area_km2 = 1e-9 # fixed-mode minimum area threshold
raw_ts_panel_b_crh_layer = "midlevel" # "column" or "midlevel"; midlevel = 400-800 hPa

# Smoothing
# raw_ts_smoothing_method = "gaussian" # "gaussian" or "running_mean"
raw_ts_smoothing_method = "running_mean" # "gaussian" or "running_mean"
raw_ts_gaussian_sigma = 1.0 # six-hourly samples; use 0 for no smoothing
raw_ts_running_mean_window = 5 # six-hourly samples; use 1 for no smoothing
raw_ts_running_mean_passes = 1 # number of passes

def running_mean(values, window_size, passes=1):
    """Apply a centered running mean one or more times."""
    if not isinstance(window_size, (int, np.integer)) or window_size < 1:
        raise ValueError("Running-mean window size must be a positive integer")
    if not isinstance(passes, (int, np.integer)) or passes < 1:
        raise ValueError("Running-mean passes must be a positive integer")

    smoothed = pd.Series(np.asarray(values, dtype=float))
    for _ in range(passes):
        smoothed = smoothed.rolling(
            window=window_size,
            center=True,
            min_periods=1,
        ).mean()
    return smoothed.to_numpy()

def gaussian_smooth(values, sigma, mode="reflect", truncate=4.0):
    """Smooth a one-dimensional series with a Gaussian kernel."""
    if not isinstance(sigma, (int, float, np.integer, np.floating)):
        raise ValueError("Gaussian sigma must be a non-negative number")
    if not np.isfinite(sigma) or sigma < 0:
        raise ValueError("Gaussian sigma must be a non-negative number")

    values = np.asarray(values, dtype=float)
    if sigma == 0:
        return values.copy()
    return gaussian_filter1d(
        values,
        sigma=sigma,
        mode=mode,
        truncate=truncate,
    )

def smooth_raw_timeseries(values):
    """Apply the smoothing method selected in the settings cell."""
    if raw_ts_smoothing_method == "gaussian":
        return gaussian_smooth(values, sigma=raw_ts_gaussian_sigma)
    if raw_ts_smoothing_method == "running_mean":
        return running_mean(
            values,
            window_size=raw_ts_running_mean_window,
            passes=raw_ts_running_mean_passes,
        )
    raise ValueError(
        "raw_ts_smoothing_method must be 'gaussian' or 'running_mean'"
    )

pclass_names = ['DC', 'CG', 'SC', 'ST', 'AN', 'DSA']
pclass_names_long = ['DeepC', 'Congest', 'Shallow', 'Stratiform', 'Anvil', 'DSA']
nclass = len(pclass_names)

# TC analysis latitude range
TC_LAT_MIN = 0
TC_LAT_MAX = 35.0
TC_ANALYSIS_START = pd.Timestamp("2000-05-01 00:00:00")
TC_ANALYSIS_END = pd.Timestamp("2000-05-11 00:00:00")
TC_TIME_FREQUENCY = "6h"

### Read and process pclass cluster data

In [ ]:
# Classifications to read/process
# raw_ts_class_indices = [0, 3, 4, 5]
# raw_ts_class_indices = [0, 5]
raw_ts_class_indices = [0, 3, 4, 5]
raw_ts_expected_snapshot_count = 40

raw_ts_panel_b_crh_layers = {"column", "midlevel"}
if raw_ts_panel_b_crh_layer not in raw_ts_panel_b_crh_layers:
    raise ValueError(
        "raw_ts_panel_b_crh_layer must be 'column' or 'midlevel'"
    )

raw_ts_area_threshold_methods = {"fixed", "percentile"}
if raw_ts_area_threshold_method not in raw_ts_area_threshold_methods:
    raise ValueError(
        "raw_ts_area_threshold_method must be 'fixed' or 'percentile'"
    )
if not np.isfinite(raw_ts_min_area_km2) or raw_ts_min_area_km2 <= 0:
    raise ValueError("The fixed raw time-series area threshold must be positive")
if len(raw_ts_area_percentile_range) != 2:
    raise ValueError("The area percentile range must contain two bounds")
raw_ts_lower_percentile, raw_ts_upper_percentile = (
    raw_ts_area_percentile_range
)
if (
    not np.isfinite(raw_ts_lower_percentile)
    or not 0 <= raw_ts_lower_percentile <= 100
):
    raise ValueError("The lower area percentile must be between 0 and 100")
if not (
    np.isposinf(raw_ts_upper_percentile)
    or (
        np.isfinite(raw_ts_upper_percentile)
        and raw_ts_lower_percentile < raw_ts_upper_percentile <= 100
    )
):
    raise ValueError(
        "The upper area percentile must exceed the lower bound and be at "
        "most 100, or be np.inf"
    )
if not set(raw_ts_class_indices).issubset(range(nclass)):
    raise ValueError("The raw time-series p-class selection is invalid")
if len(raw_ts_class_indices) != len(set(raw_ts_class_indices)):
    raise ValueError("The raw time-series p-class selection contains duplicates")

raw_ts_required_columns = {
    "lat_center", "area", "mu_mean", "md_mean", "crh_mean",
    "crh_mid_mean",
    "experiment", "time", "class_index",
}
raw_ts_value_columns = [
    "pe", "mu_area_mean", "md_abs_area_mean", "crh_area_mean",
    "crh_mid_area_mean",
    "area_mean_km2", "cluster_length_scale_km",
    "area_p95_km2", "area_total_million_km2",
]


def raw_ts_area_weighted_mean(values, weights):
    """Return an area-weighted mean over finite, positive-weight rows."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan, 0
    return np.average(values[valid], weights=weights[valid]), valid.sum()


def raw_ts_area_selection_mask(df, *, experiment):
    """Select clusters and report effective area bounds by p-class."""
    area_km2 = df["area"].astype(float) / 1e6
    base_mask = (
        df["lat_center"].between(*raw_ts_lat_bounds)
        & df["class_index"].isin(raw_ts_class_indices)
    )
    candidate_areas = area_km2.loc[base_mask].to_numpy(dtype=float)
    if candidate_areas.size == 0:
        raise ValueError(f"No area-threshold candidates for {experiment}")
    if not np.isfinite(candidate_areas).all() or (candidate_areas <= 0).any():
        raise ValueError(
            f"{experiment} area-threshold candidates must be finite and positive"
        )

    selection_mask = pd.Series(False, index=df.index, dtype=bool)
    threshold_records = []
    for class_index in raw_ts_class_indices:
        class_mask = base_mask & (df["class_index"] == class_index)
        class_areas = area_km2.loc[class_mask].to_numpy(dtype=float)
        if class_areas.size == 0:
            raise ValueError(
                f"No area-threshold candidates for {experiment}, "
                f"p-class {class_index}"
            )

        if raw_ts_area_threshold_method == "fixed":
            lower_area_km2 = float(raw_ts_min_area_km2)
            upper_area_km2 = np.inf
        else:
            lower_area_km2 = np.percentile(
                class_areas,
                raw_ts_lower_percentile,
            )
            upper_area_km2 = (
                np.inf
                if np.isposinf(raw_ts_upper_percentile)
                else np.percentile(class_areas, raw_ts_upper_percentile)
            )

        class_selection = (
            class_mask
            & area_km2.ge(lower_area_km2)
            & area_km2.le(upper_area_km2)
        )
        selection_mask |= class_selection
        threshold_records.append(
            {
                "experiment": experiment,
                "class_index": class_index,
                "class_name": pclass_names[class_index],
                "threshold_method": raw_ts_area_threshold_method,
                "lower_area_km2": lower_area_km2,
                "upper_area_km2": upper_area_km2,
                "clusters_available": class_areas.size,
                "clusters_retained": int(class_selection.sum()),
            }
        )

    return selection_mask, threshold_records


raw_ts_stat_records = []
raw_ts_selection_records = []
raw_ts_area_threshold_records = []
raw_ts_times_by_experiment = {}

# Reduce each raw experiment completely before loading the next one.
for experiment in exp_names:
    pickle_file_raw = f"{pickle_path}/pclass_cluster_{experiment}.pickle"
    with open(pickle_file_raw, "rb") as file_handle:
        df_raw_ts = pickle.load(file_handle)

    missing_columns = raw_ts_required_columns.difference(df_raw_ts.columns)
    if missing_columns:
        raise ValueError(
            f"{pickle_file_raw} is missing columns: {sorted(missing_columns)}"
        )
    file_experiments = set(df_raw_ts["experiment"].dropna().unique())
    if file_experiments != {experiment}:
        raise ValueError(
            f"Expected only {experiment} in {pickle_file_raw}; "
            f"found {file_experiments}"
        )

    if pd.api.types.is_datetime64_any_dtype(df_raw_ts["time"]):
        raw_ts_times = pd.to_datetime(df_raw_ts["time"])
    else:
        raw_ts_times = pd.to_datetime(
            df_raw_ts["time"]
            .str.replace("_", " ")
            .str.replace(".", ":", regex=False),
            format="%Y-%m-%d %H:%M:%S",
        )

    experiment_times = pd.DatetimeIndex(np.sort(raw_ts_times.unique()))
    if len(experiment_times) != raw_ts_expected_snapshot_count:
        raise ValueError(
            f"Expected {raw_ts_expected_snapshot_count} timestamps for "
            f"{experiment}; found {len(experiment_times)}"
        )
    expected_times = pd.date_range(
        experiment_times[0],
        periods=raw_ts_expected_snapshot_count,
        freq="6h",
    )
    if not experiment_times.equals(expected_times):
        raise ValueError(f"{experiment} timestamps are not complete and six-hourly")
    raw_ts_times_by_experiment[experiment] = experiment_times

    selection_mask, experiment_threshold_records = (
        raw_ts_area_selection_mask(df_raw_ts, experiment=experiment)
    )
    raw_ts_area_threshold_records.extend(experiment_threshold_records)
    selected_columns = [
        "class_index", "area", "mu_mean", "md_mean", "crh_mean",
        "crh_mid_mean",
    ]
    df_raw_ts_selected = df_raw_ts.loc[
        selection_mask, selected_columns
    ].copy()
    df_raw_ts_selected["time"] = raw_ts_times.loc[
        selection_mask
    ].to_numpy()
    df_raw_ts_selected["area_km2"] = df_raw_ts_selected["area"] / 1e6
    if df_raw_ts_selected.empty:
        raise ValueError(f"No raw time-series clusters selected for {experiment}")

    grouped = df_raw_ts_selected.groupby(
        ["time", "class_index"],
        observed=True,
        sort=True,
    )
    expected_group_count = len(experiment_times) * len(raw_ts_class_indices)
    if grouped.ngroups != expected_group_count:
        raise ValueError(
            f"Expected {expected_group_count} populated time/class groups for "
            f"{experiment}; found {grouped.ngroups}"
        )

    for (time, class_index), group in grouped:
        area_weights = group["area"].to_numpy(dtype=float)
        area_km2 = group["area_km2"].to_numpy(dtype=float)
        mu_values = group["mu_mean"].to_numpy(dtype=float)
        md_values = group["md_mean"].to_numpy(dtype=float)

        mu_area_mean, mu_sample_count = raw_ts_area_weighted_mean(
            mu_values,
            area_weights,
        )
        md_abs_area_mean, md_sample_count = raw_ts_area_weighted_mean(
            np.abs(md_values),
            area_weights,
        )
        crh_area_mean, crh_sample_count = raw_ts_area_weighted_mean(
            group["crh_mean"],
            area_weights,
        )
        crh_mid_area_mean, crh_mid_sample_count = (
            raw_ts_area_weighted_mean(
                group["crh_mid_mean"],
                area_weights,
            )
        )

        common_flux_support = (
            np.isfinite(mu_values)
            & np.isfinite(md_values)
            & np.isfinite(area_weights)
            & (area_weights > 0)
        )
        flux_sample_count = common_flux_support.sum()
        if flux_sample_count:
            mu_bulk = np.average(
                mu_values[common_flux_support],
                weights=area_weights[common_flux_support],
            )
            md_bulk = np.average(
                md_values[common_flux_support],
                weights=area_weights[common_flux_support],
            )
            pe_bulk = (
                np.nan
                if not np.isfinite(mu_bulk) or mu_bulk <= 0
                else 1 - abs(md_bulk) / mu_bulk
            )
        else:
            mu_bulk = np.nan
            md_bulk = np.nan
            pe_bulk = np.nan

        finite_areas = area_km2[np.isfinite(area_km2)]
        if finite_areas.size == 0:
            raise ValueError(
                f"No finite areas for {experiment}, {time}, p-class {class_index}"
            )
        area_mean_km2 = finite_areas.mean()
        cluster_length_scale_km = np.sqrt(area_mean_km2)

        raw_ts_stat_records.append(
            {
                "experiment": experiment,
                "time": time,
                "class_index": class_index,
                "cluster_count": len(group),
                "flux_sample_count": flux_sample_count,
                "mu_sample_count": mu_sample_count,
                "md_sample_count": md_sample_count,
                "crh_sample_count": crh_sample_count,
                "crh_mid_sample_count": crh_mid_sample_count,
                "mu_bulk": mu_bulk,
                "md_bulk": md_bulk,
                "pe": pe_bulk,
                "mu_area_mean": mu_area_mean,
                "md_abs_area_mean": md_abs_area_mean,
                "crh_area_mean": crh_area_mean,
                "crh_mid_area_mean": crh_mid_area_mean,
                "area_mean_km2": area_mean_km2,
                "cluster_length_scale_km": cluster_length_scale_km,
                "area_p95_km2": np.percentile(finite_areas, 95),
                "area_total_million_km2": finite_areas.sum() / 1e6,
            }
        )

    raw_ts_selection_records.append(
        {
            "experiment": experiment,
            "start_time": experiment_times[0],
            "end_time": experiment_times[-1],
            "snapshot_count": len(experiment_times),
            "clusters_retained": len(df_raw_ts_selected),
        }
    )

    del (
        df_raw_ts,
        raw_ts_times,
        selection_mask,
        experiment_threshold_records,
        df_raw_ts_selected,
        grouped,
    )
    gc.collect()

reference_times = raw_ts_times_by_experiment[exp_names[0]]
for experiment in exp_names[1:]:
    if not raw_ts_times_by_experiment[experiment].equals(reference_times):
        raise ValueError(
            f"Raw time-series timestamps for {experiment} do not match CTL"
        )

df_raw_timeseries_stats = pd.DataFrame(raw_ts_stat_records)
raw_ts_stat_keys = ["experiment", "time", "class_index"]
if df_raw_timeseries_stats.duplicated(raw_ts_stat_keys).any():
    raise ValueError("Raw time-series statistics contain duplicate keys")
expected_stat_rows = (
    len(exp_names)
    * len(reference_times)
    * len(raw_ts_class_indices)
)
if len(df_raw_timeseries_stats) != expected_stat_rows:
    raise ValueError(
        f"Expected {expected_stat_rows} raw time-series statistic rows; "
        f"found {len(df_raw_timeseries_stats)}"
    )

df_raw_timeseries_long = df_raw_timeseries_stats.melt(
    id_vars=raw_ts_stat_keys,
    value_vars=raw_ts_value_columns,
    var_name="variable",
    value_name="value",
)
df_raw_timeseries_difference = (
    df_raw_timeseries_long
    .pivot(
        index=["time", "class_index", "variable"],
        columns="experiment",
        values="value",
    )
    .reset_index()
    .sort_values(["class_index", "variable", "time"])
    .reset_index(drop=True)
)
df_raw_timeseries_difference.columns.name = None

missing_experiments = set(exp_names).difference(
    df_raw_timeseries_difference.columns
)
if missing_experiments:
    raise ValueError(
        f"Missing raw time-series experiments: {sorted(missing_experiments)}"
    )
if df_raw_timeseries_difference[exp_names].isna().any().any():
    raise ValueError("Raw time-series statistics do not have complete pairs")

# Smooth each experiment independently before calculating differences.
raw_ts_smoothing_keys = ["class_index", "variable"]
raw_ts_smoothed_columns = {}
for experiment in exp_names:
    smoothed_column = f"{experiment}_smoothed"
    raw_ts_smoothed_columns[experiment] = smoothed_column
    df_raw_timeseries_difference[smoothed_column] = (
        df_raw_timeseries_difference
        .groupby(raw_ts_smoothing_keys, sort=False)[experiment]
        .transform(
            lambda series: smooth_raw_timeseries(series.to_numpy())
        )
    )

ctl_smoothed_column = raw_ts_smoothed_columns["CTL"]
off_lw_smoothed_column = raw_ts_smoothed_columns["CLIM_RAD_LW"]
zero_ctl = df_raw_timeseries_difference[ctl_smoothed_column] == 0
if zero_ctl.any():
    zero_keys = df_raw_timeseries_difference.loc[
        zero_ctl,
        ["time", "class_index", "variable"],
    ]
    raise ValueError(
        "Cannot calculate percent differences with zero smoothed CTL "
        "values:\n"
        f"{zero_keys.to_string(index=False)}"
    )
df_raw_timeseries_difference["percent_difference"] = 100 * (
    df_raw_timeseries_difference[off_lw_smoothed_column]
    - df_raw_timeseries_difference[ctl_smoothed_column]
) / df_raw_timeseries_difference[ctl_smoothed_column]
df_raw_timeseries_difference["Hours"] = (
    (
        df_raw_timeseries_difference["time"]
        - df_raw_timeseries_difference["time"].min()
    ).dt.total_seconds()
    / 3600
    + 6
)
df_raw_timeseries_difference["Days"] = (
    df_raw_timeseries_difference["Hours"] / 24
)

raw_ts_difference_keys = ["time", "class_index", "variable"]
if df_raw_timeseries_difference.duplicated(raw_ts_difference_keys).any():
    raise ValueError("Raw time-series differences contain duplicate keys")
expected_difference_rows = (
    len(reference_times)
    * len(raw_ts_class_indices)
    * len(raw_ts_value_columns)
)
if len(df_raw_timeseries_difference) != expected_difference_rows:
    raise ValueError(
        f"Expected {expected_difference_rows} raw time-series differences; "
        f"found {len(df_raw_timeseries_difference)}"
    )

df_raw_timeseries_selection_summary = pd.DataFrame(
    raw_ts_selection_records
)
df_raw_timeseries_area_thresholds = pd.DataFrame(
    raw_ts_area_threshold_records
)
print(df_raw_timeseries_area_thresholds.to_string(index=False))
print(df_raw_timeseries_selection_summary.to_string(index=False))
print(
    f"Raw time-series rows: {len(df_raw_timeseries_stats):,}; "
    f"paired differences: {len(df_raw_timeseries_difference):,}"
)

### Plotting functions

In [ ]:
# ============================
# Helper functions for figure
# ============================

def read_TC_tracks(
    exp_name,
    tracker="TE",
    lat_min=0,
    lat_max=35,
    start_time=TC_ANALYSIS_START,
    end_time=TC_ANALYSIS_END,
):
    """Read and validate one experiment's TC track file."""
    if tracker != "TE":
        raise ValueError(f"Unsupported TC tracker: {tracker}")
    if lat_min > lat_max:
        raise ValueError("TC latitude minimum must not exceed the maximum")

    filename = f"{data_main}tc_tracks_{exp_name}.txt"
    df_tracks = pd.read_csv(
        filename,
        header=0,
        skipinitialspace=True,
    )
    required_columns = {
        "track_id", "year", "month", "day", "hour",
        "i", "lon", "lat", "slp", "wind",
    }
    missing_columns = required_columns.difference(df_tracks.columns)
    if missing_columns:
        raise ValueError(
            f"{filename} is missing columns: {sorted(missing_columns)}"
        )
    if not np.isfinite(df_tracks["wind"].to_numpy(dtype=float)).all():
        raise ValueError(f"{filename} contains non-finite wind values")

    df_tracks["datetime"] = pd.to_datetime(
        df_tracks[["year", "month", "day", "hour"]]
    )
    if df_tracks.duplicated(["track_id", "datetime"]).any():
        raise ValueError(f"{filename} contains duplicate track/time records")

    selection = (
        df_tracks["lat"].between(lat_min, lat_max)
        & df_tracks["datetime"].between(start_time, end_time)
    )
    df_tracks = df_tracks.loc[selection].copy()
    if df_tracks.empty:
        raise ValueError(
            f"No {exp_name} TC tracks remain after latitude/time filtering"
        )
    return df_tracks.sort_values(["datetime", "track_id"]).reset_index(
        drop=True
    )

def cumulative_tc_counts_by_threshold(
    df,
    *,
    time_index,
    excluded_track_ids=None,
    id_col="track_id",
    time_col="datetime",
    wind_col="wind",
    thresholds=(17., 33., 50., 70.),
):
    """Count each track once, when it first reaches each wind threshold."""
    required_columns = {id_col, time_col, wind_col}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(
            f"TC data are missing columns: {sorted(missing_columns)}"
        )

    time_index = pd.DatetimeIndex(time_index)
    if time_index.empty or not time_index.is_monotonic_increasing:
        raise ValueError("TC time index must be nonempty and increasing")
    if time_index.has_duplicates:
        raise ValueError("TC time index must not contain duplicates")

    d = df[[id_col, time_col, wind_col]].copy()
    d[time_col] = pd.to_datetime(d[time_col])
    if excluded_track_ids is not None:
        d = d.loc[~d[id_col].isin(excluded_track_ids)]

    out = {}
    for threshold in thresholds:
        if not np.isfinite(threshold) or threshold <= 0:
            raise ValueError("TC wind thresholds must be finite and positive")
        first_hit = (
            d.loc[d[wind_col] >= threshold]
            .groupby(id_col, observed=True)[time_col]
            .min()
        )
        first_hit_counts = (
            first_hit.dt.floor(TC_TIME_FREQUENCY)
            .value_counts()
            .reindex(time_index, fill_value=0)
            .sort_index()
        )
        out[threshold] = first_hit_counts.cumsum().astype(int)

    cumulative_counts = pd.DataFrame(out, index=time_index)
    cumulative_counts.index.name = time_col
    return cumulative_counts

def validate_cumulative_tc_counts(counts, thresholds, label):
    """Validate monotonic and nested cumulative threshold counts."""
    if counts.empty:
        raise ValueError(f"{label} TC counts are empty")
    if counts.diff().iloc[1:].lt(0).any().any():
        raise ValueError(f"{label} TC counts are not monotonic")
    for lower_threshold, higher_threshold in zip(
        thresholds[:-1],
        thresholds[1:],
    ):
        if (counts[lower_threshold] < counts[higher_threshold]).any():
            raise ValueError(
                f"{label} counts are not nested across thresholds"
            )

### Combined PClass Cluster statistics and TC statistics

In [ ]:
# =============================================================================
# Figure: 3 columns × 2 rows
#   Top row:    OffLW - CTL cluster-statistic time series
#   Bottom row: Cumulative TCs, Hurricanes, Major Hurricanes
# =============================================================================

fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(11, 7.5),
                        gridspec_kw={'hspace': 0.60},
                        constrained_layout=False)
fig.subplots_adjust(left=0.08, right=0.96, top=0.86, bottom=0.08)

# colors = sns.color_palette('muted', n_colors=2)
sns.set_theme(style="ticks", font_scale=1.1)
colors = ['black', 'dodgerblue']

lw = 1.6


# -----------------------------------------------------------------
# TOP ROW — MCS CCDFs (epsilon, area, lifetime)
# -----------------------------------------------------------------


raw_ts_panel_b_options = {
    "column": ("crh_area_mean", "(b) Area-weighted column CRH"),
    "midlevel": (
        "crh_mid_area_mean",
        r"(b) $r_{mid}$",
    ),
}
raw_ts_panel_b_variable, raw_ts_panel_b_title = (
    raw_ts_panel_b_options[raw_ts_panel_b_crh_layer]
)

raw_ts_variable_metadata = {
    "pe": (r"(a) $\epsilon$", f"% change from CTL"),
    # "mu_area_mean": (
    #     r"(b) Area-weighted $M_u$",
    #     r"kg m$^{-2}$ s$^{-1}$",
    # ),
    # "md_abs_area_mean": (
    #     r"(c) Area-weighted $|M_d|$",
    #     r"kg m$^{-2}$ s$^{-1}$",
    # ),
    raw_ts_panel_b_variable: (raw_ts_panel_b_title, "%"),
    "cluster_length_scale_km": (
        "(c) Mean size",
        "%",
    ),
    # "area_p95_km2": (
    #     "(f) 95th-percentile cluster area",
    #     r"km$^2$",
    # ),
    # "area_total_million_km2": (
    #     "(g) Total cluster area",
    #     r"$10^6$ km$^2$",
    # ),
}

# Combined view: all selected p-classes in each variable panel.

linestyle=['-','--']
combined_figure_class_indices = [5]
raw_ts_class_colors = dict(
    zip(
        raw_ts_class_indices,
        sns.color_palette("muted", n_colors=len(raw_ts_class_indices)),
    )
)

for iax, (variable, (title, units)) in zip(
    ax[0,:],
    raw_ts_variable_metadata.items(),
):
    for iclass, class_target in enumerate(combined_figure_class_indices):
        df_plot = df_raw_timeseries_difference[
            (df_raw_timeseries_difference["class_index"] == class_target)
            & (df_raw_timeseries_difference["variable"] == variable)
        ].sort_values("time")
        if len(df_plot) != len(reference_times):
            raise ValueError(
                f"Incomplete {variable} time series for p-class {class_target}"
            )

        iax.plot(
            df_plot["Days"],
            df_plot["percent_difference"],
            # color=raw_ts_class_colors[class_target],
            color=colors[1],
            linestyle="-",
            linewidth=1.8,
            # marker="o",
            # markersize=2.5,
            label=pclass_names_long[class_target],
        )
        # for i_exp in range(ntest):
        #     iax.plot(
        #         df_plot["Days"],
        #         df_plot[exp_names[i_exp]],
        #         color=colors[i_exp],
        #         linestyle=linestyle[iclass],
        #         linewidth=lw,
        #         marker="o",
        #         markersize=2.5,
        #         label=pclass_names_long[class_target],
        #     )

    iax.axhline(0, color="black", linewidth=0.8, zorder=0)
    iax.set_title(title)
    if variable == "pe":
        iax.set_ylabel(units)
    iax.set_xlim(0, 10)
    iax.xaxis.set_major_locator(ticker.MultipleLocator(2))
    sns.despine(ax=iax, offset=5)

ax[0,1].set_xlabel("Day")
legend_handles, legend_labels = ax[0,0].get_legend_handles_labels()
# fig.legend(
#     legend_handles,
#     legend_labels,
#     loc="upper center",
#     bbox_to_anchor=(0.5, 0.935),
#     ncol=len(raw_ts_class_indices),
#     frameon=False,
#     handlelength=2.0,
#     handletextpad=0.5,
#     columnspacing=1.5,
#     borderaxespad=0,
# )
# fig.suptitle(
#     "Raw-cluster p-classes: CTL - Off-LWCRF\n"
#     f"Clusters >= {raw_ts_min_area_km2:g} km$^2$; "
#     f"{raw_ts_lat_bounds[0]}-{raw_ts_lat_bounds[1]}° latitude",
#     y=0.985,
# )

# Despine top row
# for i, iax in enumerate(ax.flatten()):
#     sns.despine(offset=5, ax=iax, top=True, right=True)
#     if i != 0 and i != 3:
#         iax.set_ylabel('')
#         iax.set_yticklabels([])


# -----------------------------------------------------------------
# BOTTOM ROW — Cumulative TC counts (replicates cell 10 of
#              analysis_of_TC_statistics.ipynb)
# -----------------------------------------------------------------


thresholds = (17., 33., 50.)
panel_labels = [
    '(d) TCs (≥17 m/s)',
    '(e) Hurricanes (≥33 m/s)',
    '(f) Major Hurricanes (≥50 m/s)',
]

tc_time_index = pd.date_range(
    TC_ANALYSIS_START,
    TC_ANALYSIS_END,
    freq=TC_TIME_FREQUENCY,
)
tc_days = (
    (tc_time_index - TC_ANALYSIS_START).total_seconds() / 86400.0
)
tc_tracks = {}
tc_counts = {}
tc_initial_times = {}
tc_initial_track_ids = {}
tc_count_summary_records = []

for exp in exp_names:
    df_tc = read_TC_tracks(
        exp,
        tracker="TE",
        lat_min=TC_LAT_MIN,
        lat_max=TC_LAT_MAX,
    )
    initial_time = df_tc["datetime"].min()
    initial_track_ids = frozenset(
        df_tc.loc[df_tc["datetime"] == initial_time, "track_id"]
    )
    tc_tracks[exp] = df_tc
    tc_initial_times[exp] = initial_time
    tc_initial_track_ids[exp] = initial_track_ids
    tc_counts[exp] = {
        "including_initial": cumulative_tc_counts_by_threshold(
            df_tc,
            time_index=tc_time_index,
            thresholds=thresholds,
        ),
        "excluding_initial": cumulative_tc_counts_by_threshold(
            df_tc,
            time_index=tc_time_index,
            excluded_track_ids=initial_track_ids,
            thresholds=thresholds,
        ),
    }

    for count_type, counts in tc_counts[exp].items():
        validate_cumulative_tc_counts(
            counts,
            thresholds,
            label=f"{exp} {count_type}",
        )
        eligible_tracks = df_tc
        if count_type == "excluding_initial":
            eligible_tracks = df_tc.loc[
                ~df_tc["track_id"].isin(initial_track_ids)
            ]
        track_max_wind = eligible_tracks.groupby("track_id")["wind"].max()
        for threshold in thresholds:
            direct_count = int((track_max_wind >= threshold).sum())
            cumulative_count = int(counts[threshold].iloc[-1])
            if cumulative_count != direct_count:
                raise ValueError(
                    f"{exp} {count_type} {threshold:g} m/s count mismatch: "
                    f"{cumulative_count} cumulative versus {direct_count} direct"
                )
            tc_count_summary_records.append(
                {
                    "experiment": exp,
                    "count_type": count_type,
                    "threshold_m_s": threshold,
                    "final_count": cumulative_count,
                    "initial_time": initial_time,
                    "initial_track_count": len(initial_track_ids),
                }
            )

df_tc_count_summary = pd.DataFrame(tc_count_summary_records)
print(df_tc_count_summary.to_string(index=False))

tc_lw_including = 1.8
# tc_lw_excluding = 0.9
tc_lw_excluding = lw

for i, thr in enumerate(thresholds):
    iax = ax[1, i]
    for iexp, exp in enumerate(exp_names):
        plot_mask = tc_time_index >= tc_initial_times[exp]
        plot_days = tc_days[plot_mask]
        including_initial = tc_counts[exp]["including_initial"].loc[
            plot_mask,
            thr,
        ]
        excluding_initial = tc_counts[exp]["excluding_initial"].loc[
            plot_mask,
            thr,
        ]
        iax.plot(
            plot_days,
            including_initial,
            color=colors[iexp],
            linewidth=tc_lw_including,
            zorder=2,
            label=f"{exp_tags[iexp]}",# incl. initial",
        )
        # iax.plot(
        #     plot_days,
        #     excluding_initial,
        #     color=colors[iexp],
        #     linewidth=tc_lw_excluding,
        #     zorder=3,
        #     label=f"{exp_tags[iexp]}",# excl. initial",
        # )

    iax.set_title(panel_labels[i])
    iax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    sns.despine(ax=iax, offset=5, right=True, top=True)
    panel_max = max(
        tc_counts[exp]["including_initial"][thr].max()
        for exp in exp_names
    )
    iax.set_xlim(tc_days[0], tc_days[-1])
    # iax.set_ylim(-0.5, max(1, panel_max + 1))
    # iax.set_ylim(-1, 14)
    iax.set_ylim(-1, 20)
    if i > 0:
        iax.set_ylabel('')
        iax.set_yticklabels([])

ax[1, 1].set_xlabel("Day")
ax[1, 0].set_ylabel("#")
handles, labels = ax[1, 0].get_legend_handles_labels()
ax[1, 2].legend(
    handles,
    labels,
    frameon=False,
    # fontsize=8.5,
    loc="upper left",
)

# # Despine bottom row
# for i, iax in enumerate(ax.flatten()):
#     sns.despine(offset=5, ax=iax, top=True, right=True)
#     # if i != 0 and i != 3:
#     if i > 2:
#         iax.set_ylabel('')
#         iax.set_yticklabels([])

# ---------------------------------
# Add titles to top and bottom rows
# ---------------------------------

# plt.suptitle(suptitle)
y_offset = 0.12
fig.text(
            0.5, 0.975,
            f'Aquaplanet Cluster Statistics',
            ha='center', va='top', fontsize=14, fontweight='bold')
ax[1,1].annotate('Cumulative TC Counts',
            xy=(0.5, 1+y_offset), xycoords='axes fraction',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

# plt.savefig(f"{figdir}/mpas_mcs_tc_comparison.png", dpi=400, facecolor='white',
#         bbox_inches='tight')
plt.savefig(f"{figdir}/mpas_mcs_tc_comparison.pdf", bbox_inches='tight')
plt.show()
plt.close()

### Cluster area by class

In [ ]:
# Mean cluster size for DeepC, Stratiform, Anvil, and DSA.
mean_size_class_indices = [0, 3, 4, 5]
fig_mean_size, ax_mean_size = plt.subplots(figsize=(7.5, 4.5))
mean_size_colors = sns.color_palette(
    "muted", n_colors=len(mean_size_class_indices)
)

for class_index, class_color in zip(
    mean_size_class_indices, mean_size_colors
):
    df_mean_size = df_raw_timeseries_difference[
        (df_raw_timeseries_difference["class_index"] == class_index)
        & (
            df_raw_timeseries_difference["variable"]
            == "cluster_length_scale_km"
        )
    ].sort_values("time")
    if len(df_mean_size) != len(reference_times):
        raise ValueError(
            f"Incomplete mean-size time series for p-class {class_index}"
        )

    ax_mean_size.plot(
        df_mean_size["Days"],
        df_mean_size["percent_difference"],
        color=class_color,
        linewidth=1.8,
        label=pclass_names_long[class_index],
    )

ax_mean_size.axhline(0, color="black", linewidth=0.8, zorder=0)
ax_mean_size.set(
    title="Mean cluster size",
    xlabel="Day",
    ylabel="% change from CTL",
    xlim=(0, 10),
)
ax_mean_size.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax_mean_size.legend(frameon=False, ncol=2)
sns.despine(ax=ax_mean_size, offset=5)
plt.savefig(f"{figdir}/mpas_cluster_mean_size_comparison.pdf", bbox_inches='tight')
plt.show()
plt.close()

### TC track starting locations

In [ ]:
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# Use the earliest retained point for each track in the analysis period.
tc_start_location_frames = []
for exp in exp_names:
    df_tc_starts = (
        tc_tracks[exp]
        .sort_values(["track_id", "datetime"])
        .drop_duplicates("track_id", keep="first")
        .copy()
    )
    if df_tc_starts.empty:
        raise ValueError(f"No TC track starting locations found for {exp}")

    start_coordinates = df_tc_starts[["lon", "lat"]].to_numpy(dtype=float)
    if not np.isfinite(start_coordinates).all():
        raise ValueError(f"{exp} TC starting locations contain non-finite values")
    if not df_tc_starts["lat"].between(-90, 90).all():
        raise ValueError(f"{exp} TC starting latitudes fall outside -90 to 90")

    df_tc_starts["experiment"] = exp
    df_tc_starts["plot_lon"] = (
        (df_tc_starts["lon"] + 180) % 360
    ) - 180
    tc_start_location_frames.append(df_tc_starts)

df_tc_start_locations = pd.concat(
    tc_start_location_frames,
    ignore_index=True,
)

tc_data_crs = ccrs.PlateCarree()
tc_map_projection = ccrs.PlateCarree(central_longitude=180)
fig_tc_map = plt.figure(figsize=(11, 5.5))
ax_tc_map = fig_tc_map.add_subplot(1, 1, 1, projection=tc_map_projection)
ax_tc_map.set_global()
ax_tc_map.set_facecolor("aliceblue")

tc_map_gridlines = ax_tc_map.gridlines(
    crs=tc_data_crs,
    draw_labels=True,
    linewidth=0.6,
    color="0.55",
    alpha=0.7,
    linestyle=":",
)
tc_map_gridlines.top_labels = False
tc_map_gridlines.right_labels = False
tc_map_gridlines.xlocator = ticker.FixedLocator(np.arange(-180, 181, 60))
tc_map_gridlines.ylocator = ticker.FixedLocator(np.arange(-60, 61, 30))
tc_map_gridlines.xformatter = LONGITUDE_FORMATTER
tc_map_gridlines.yformatter = LATITUDE_FORMATTER

tc_start_markers = ("o", "^", "s", "D")
for iexp, exp in enumerate(exp_names):
    df_plot = df_tc_start_locations.loc[
        df_tc_start_locations["experiment"] == exp
    ]
    ax_tc_map.scatter(
        df_plot["plot_lon"],
        df_plot["lat"],
        transform=tc_data_crs,
        s=48,
        marker=tc_start_markers[iexp % len(tc_start_markers)],
        color=colors[iexp],
        edgecolor="white",
        linewidth=0.7,
        alpha=0.9,
        zorder=3,
        label=f"{exp_tags[iexp]} (n={len(df_plot)})",
    )

ax_tc_map.set_title("TC track starting locations")
ax_tc_map.legend(
    loc="lower center",
    ncol=len(exp_names),
    frameon=False,
)
fig_tc_map.tight_layout()
plt.show()
plt.close(fig_tc_map)

### TC track starting-latitude distribution

In [ ]:
# Normalize each experiment independently so their distributions are comparable.
tc_latitude_bin_width = 5.0
tc_latitude_bins = np.append(
    np.arange(TC_LAT_MIN, TC_LAT_MAX, tc_latitude_bin_width),
    TC_LAT_MAX,
)

fig_tc_latitude, ax_tc_latitude = plt.subplots(figsize=(7.5, 4.5))
tc_latitude_linestyles = ("-", "--", ":", "-.")

for iexp, exp in enumerate(exp_names):
    starting_latitudes = df_tc_start_locations.loc[
        df_tc_start_locations["experiment"] == exp,
        "lat",
    ].to_numpy(dtype=float)
    if starting_latitudes.size == 0:
        raise ValueError(f"No TC starting latitudes found for {exp}")
    if not np.isfinite(starting_latitudes).all():
        raise ValueError(f"{exp} TC starting latitudes contain non-finite values")
    if not (
        (starting_latitudes >= TC_LAT_MIN)
        & (starting_latitudes <= TC_LAT_MAX)
    ).all():
        raise ValueError(f"{exp} TC starting latitudes fall outside the analysis range")

    ax_tc_latitude.hist(
        starting_latitudes,
        bins=tc_latitude_bins,
        weights=np.full(starting_latitudes.size, 1 / starting_latitudes.size),
        histtype="step",
        linewidth=2.0,
        linestyle=tc_latitude_linestyles[
            iexp % len(tc_latitude_linestyles)
        ],
        color=colors[iexp],
        label=f"{exp_tags[iexp]} (n={starting_latitudes.size})",
    )

ax_tc_latitude.set_title("Distribution of TC track starting latitudes")
ax_tc_latitude.set_xlabel("Starting latitude (°N)")
ax_tc_latitude.set_ylabel("Fraction of tracks per 5° bin")
ax_tc_latitude.set_xlim(TC_LAT_MIN, TC_LAT_MAX)
ax_tc_latitude.set_ylim(bottom=0)
ax_tc_latitude.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax_tc_latitude.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax_tc_latitude.legend(frameon=False)
sns.despine(ax=ax_tc_latitude, offset=5)
fig_tc_latitude.tight_layout()
plt.show()
plt.close(fig_tc_latitude)